In [1]:
import pandas as pd
import re
import string
import nltk

from sklearn.model_selection import train_test_split
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mereh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mereh\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\mereh\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [ ]:
df = pd.read_csv("data/IMDb Dataset.csv")

print(df.head())


                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [3]:
print("Dataset Shape:")
print(df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nTarget Column Counts:")
print(df["sentiment"].value_counts())


Dataset Shape:
(50000, 2)

Missing Values:
review       0
sentiment    0
dtype: int64

Duplicate Rows:
418

Target Column Counts:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [4]:
df.drop_duplicates(inplace=True)

print("\nShape After Removing Duplicates:")
print(df.shape)


Shape After Removing Duplicates:
(49582, 2)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    df["review"],
    df["sentiment"],
    test_size=0.2,
    random_state=42
)

print("\nTraining Samples:", len(X_train))
print("Testing Samples:", len(X_test))


Training Samples: 39665
Testing Samples: 9917


In [6]:
# Lowercase
X_train = X_train.str.lower()

# Remove URLs
X_train = X_train.str.replace(
    r'http\S+|www\S+',
    '',
    regex=True
)

# Remove Numbers
X_train = X_train.str.replace(
    r'\d+',
    '',
    regex=True
)

# Remove Punctuation
X_train = X_train.str.replace(
    f'[{string.punctuation}]',
    '',
    regex=True
)

# Remove Extra Spaces
X_train = X_train.str.replace(
    r'\s+',
    ' ',
    regex=True
).str.strip()

print("\nCleaned Training Reviews:")
print(X_train.head())


Cleaned Training Reviews:
7837     i really liked the movie the emporers new groo...
4814     i decided to watch this movie because it has b...
35458    its very hard to say just what was going on wi...
3446     this scifi adventure is not the best and by no...
24478    around the late s animator don bluth frustrate...
Name: review, dtype: object


In [7]:
# Lowercase
X_test = X_test.str.lower()

# Remove URLs
X_test = X_test.str.replace(r'http\S+|www\S+', '', regex=True)

# Remove Numbers
X_test = X_test.str.replace(r'\d+', '', regex=True)

# Remove Punctuation
X_test = X_test.str.replace(f'[{string.punctuation}]', '', regex=True)

# Remove Extra Spaces
X_test = X_test.str.replace(r'\s+', ' ', regex=True).str.strip()

# Tokenization
test_tokens = X_test.apply(word_tokenize)


In [8]:
train_tokens = X_train.apply(word_tokenize)

print("\nTokenized Reviews:")
print(train_tokens.head())



Tokenized Reviews:
7837     [i, really, liked, the, movie, the, emporers, ...
4814     [i, decided, to, watch, this, movie, because, ...
35458    [its, very, hard, to, say, just, what, was, go...
3446     [this, scifi, adventure, is, not, the, best, a...
24478    [around, the, late, s, animator, don, bluth, f...
Name: review, dtype: object


In [9]:
all_tokens = []

for tokens in train_tokens:
    all_tokens.extend(tokens)

print("\nTotal Tokens:", len(all_tokens))
print("Vocabulary Size:", len(set(all_tokens)))



Total Tokens: 9103497
Vocabulary Size: 155437


In [10]:
stemmer = PorterStemmer()

train_stemmed = train_tokens.apply(
    lambda words: [stemmer.stem(word) for word in words]
)

print("\nStemmed Reviews:")
print(train_stemmed.head())



Stemmed Reviews:
7837     [i, realli, like, the, movi, the, empor, new, ...
4814     [i, decid, to, watch, thi, movi, becaus, it, h...
35458    [it, veri, hard, to, say, just, what, wa, go, ...
3446     [thi, scifi, adventur, is, not, the, best, and...
24478    [around, the, late, s, anim, don, bluth, frust...
Name: review, dtype: object


In [11]:
stem_tokens = []

for words in train_stemmed:
    stem_tokens.extend(words)

print("\nTotal Stemmed Tokens:", len(stem_tokens))
print("Stem Vocabulary Size:", len(set(stem_tokens)))


Total Stemmed Tokens: 9103497
Stem Vocabulary Size: 120649


In [12]:
lemmatizer = WordNetLemmatizer()

train_lemmatized = train_tokens.apply(
    lambda words: [lemmatizer.lemmatize(word) for word in words]
)

print("\nLemmatized Reviews:")
print(train_lemmatized.head())



Lemmatized Reviews:
7837     [i, really, liked, the, movie, the, emporers, ...
4814     [i, decided, to, watch, this, movie, because, ...
35458    [it, very, hard, to, say, just, what, wa, goin...
3446     [this, scifi, adventure, is, not, the, best, a...
24478    [around, the, late, s, animator, don, bluth, f...
Name: review, dtype: object


In [13]:
lemma_tokens = []

for words in train_lemmatized:
    lemma_tokens.extend(words)

print("\nTotal Lemmatized Tokens:", len(lemma_tokens))
print("Lemma Vocabulary Size:", len(set(lemma_tokens)))


Total Lemmatized Tokens: 9103497
Lemma Vocabulary Size: 145086


In [14]:
comparison = pd.DataFrame({
    "Original Review": df.loc[X_train.index, "review"],
    "Cleaned Review": X_train,
    "Tokenized": train_tokens,
    "Stemmed": train_stemmed,
    "Lemmatized": train_lemmatized
})

comparison.head()

,Original Review,Cleaned Review,Tokenized,Stemmed,Lemmatized
7837,I really liked the movie 'The Emporer's New Gr...,i really liked the movie the emporers new groo...,"[i, really, liked, the, movie, the, emporers, ...","[i, realli, like, the, movi, the, empor, new, ...","[i, really, liked, the, movie, the, emporers, ..."
4814,I decided to watch this movie because it has b...,i decided to watch this movie because it has b...,"[i, decided, to, watch, this, movie, because, ...","[i, decid, to, watch, thi, movi, becaus, it, h...","[i, decided, to, watch, this, movie, because, ..."
35458,It's very hard to say just what was going on w...,its very hard to say just what was going on wi...,"[its, very, hard, to, say, just, what, was, go...","[it, veri, hard, to, say, just, what, wa, go, ...","[it, very, hard, to, say, just, what, wa, goin..."
3446,This sci-fi adventure is not the best and by n...,this scifi adventure is not the best and by no...,"[this, scifi, adventure, is, not, the, best, a...","[thi, scifi, adventur, is, not, the, best, and...","[this, scifi, adventure, is, not, the, best, a..."
24478,"Around the late 1970's, animator Don Bluth, fr...",around the late s animator don bluth frustrate...,"[around, the, late, s, animator, don, bluth, f...","[around, the, late, s, anim, don, bluth, frust...","[around, the, late, s, animator, don, bluth, f..."


In [15]:
results = pd.DataFrame({
    "Stage": [
        "Tokenization",
        "Stemming",
        "Lemmatization"
    ],
    "Total Tokens": [
        len(all_tokens),
        len(stem_tokens),
        len(lemma_tokens)
    ],
    "Vocabulary Size": [
        len(set(all_tokens)),
        len(set(stem_tokens)),
        len(set(lemma_tokens))
    ]
})

print(results)

           Stage  Total Tokens  Vocabulary Size
0   Tokenization       9103497           155437
1       Stemming       9103497           120649
2  Lemmatization       9103497           145086


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(embeddings)

print(similarity_matrix)

In [ ]:
article = 0

scores = similarity_matrix[article]

scores[article] = -1

most_similar = scores.argmax()

print("Original Article:\n")
print(df.iloc[article]["descr"])

print("\nMost Similar Article:\n")
print(df.iloc[most_similar]["descr"])

print("\nSimilarity Score:", scores[most_similar])

In [ ]:
top5 = scores.argsort()[-5:][::-1]

print("Top 5 Similar Articles\n")

for idx in top5:
    print(df.iloc[idx]["descr"][:200], "...")
    print("Similarity:", scores[idx])
    print("-"*80)

In [ ]:
duplicates = df[df.duplicated(subset="descr", keep=False)]

print("Duplicate Articles:")

duplicates

In [ ]:
for i, row in top5.iterrows():
    print("=" * 80)
    print("Embedding Score:", row["Embedding Score"])
    print()
    print(row["descr"][:1000])